<b>Transform Customer Data

1. Remove records with NULL customer_id

2. Remove exact duplicate records

3. Remove duplicate records based on created_timestamp

4. Cast column values to the correct data types

5. Write the transformed data to the Silver schema

<b>1. Remove records with NULL customer_id



In [0]:
%sql
SELECT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL; 

<b>2. Remove exact duplicate records



In [0]:
%sql
SELECT *
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
SELECT DISTINCT * 
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
SELECT customer_id,
       MAX(created_timestamp),
       MAX(customer_name),
       MAX(date_of_birth),
       MAX(email),
       MAX(member_since),
       MAX(telephone)
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
GROUP BY customer_id
ORDER BY customer_id;


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_customers_distinct
AS
SELECT DISTINCT * 
FROM gizmobox.bronze.v_customers
WHERE customer_id IS NOT NULL
ORDER BY customer_id;

In [0]:
%sql
SELECT customer_id,
       MAX(created_timestamp) max_created_timestamp
FROM v_customers_distinct
GROUP BY customer_id;

<b>3. Remove duplicate records based on created_timestamp



In [0]:
%sql
WITH cte_max AS(
  SELECT customer_id,
        MAX(created_timestamp) max_created_timestamp
  FROM v_customers_distinct
  GROUP BY customer_id
)
SELECT t.*
  FROM v_customers_distinct t
  JOIN cte_max
    ON t.customer_id = cte_max.customer_id
    AND t.created_timestamp = cte_max.max_created_timestamp

<b>4. Cast the column values to the correct data type










In [0]:
%sql
WITH cte_max AS(
  SELECT customer_id,
        MAX(created_timestamp) max_created_timestamp
  FROM v_customers_distinct
  GROUP BY customer_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) created_timestamp,
       t.customer_id,
       t.customer_name,
       CAST(t.date_of_birth AS DATE) date_of_birth,
       t.email,
       CAST(t.member_since AS DATE) member_since,
       t.telephone
  FROM v_customers_distinct t
  JOIN cte_max m
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp

<b>5. Write Data to a Delta Table

In [0]:
%sql
CREATE TABLE gizmobox.silver.customers
AS
WITH cte_max AS(
  SELECT customer_id,
        MAX(created_timestamp) max_created_timestamp
  FROM v_customers_distinct
  GROUP BY customer_id
)
SELECT CAST(t.created_timestamp AS TIMESTAMP) created_timestamp,
       t.customer_id,
       t.customer_name,
       CAST(t.date_of_birth AS DATE) date_of_birth,
       t.email,
       CAST(t.member_since AS DATE) member_since,
       t.telephone
  FROM v_customers_distinct t
  JOIN cte_max m 
    ON t.customer_id = m.customer_id
    AND t.created_timestamp = m.max_created_timestamp;

In [0]:
%sql 
DESCRIBE EXTENDED gizmobox.silver.customers;

In [0]:
%sql
DROP TABLE gizmobox.silver.customers
